#1. Load train_feat.csv, split features X and label y
#2. train_test_split + StratifiedKFold cross-validation
#3. Run baseline on multiple models (LogisticRegression, RandomForest, XGBoost), record CV scores for each
#4. Select RandomForest (or best model) for final training + hold-out testing, output accuracy, classification_report
#5. Save trained model with joblib.dump(model, '../models/rf.joblib')
#6. Write basic performance metrics to results/metrics.txt


In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [2]:
# ── 1. Load data, split X / y ───────────────────────────────
df = pd.read_csv('../data/processed/train_feat.csv')
print(f"Data shape: {df.shape}")
print(f"Column names: {df.columns.tolist()}")

X = df.drop(columns=['Survived'])
y = df['Survived']
print(f"\nX: {X.shape}, y: {y.shape}")
print(f"y distribution:\n{y.value_counts()}")


Data shape: (891, 9)
Column names: ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']

X: (891, 8), y: (891,)
y distribution:
Survived
0    549
1    342
Name: count, dtype: int64


In [3]:
# ── 2. train_test_split ───────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # Maintain positive/negative sample ratio
)
print(f"\ntrain: {X_train.shape}, test: {X_test.shape}")



train: (712, 8), test: (179, 8)


In [4]:
# ── 3. Multi-model CV baseline ─────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':            XGBClassifier(n_estimators=100, random_state=42,
                                        eval_metric='logloss', verbosity=0),
}

cv_results = {}
print("\n── CV Results (5-Fold Accuracy)──")
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train,
                             cv=skf, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:22s}  mean={scores.mean():.4f}  std={scores.std():.4f}")



── CV Results (5-Fold Accuracy)──
LogisticRegression      mean=0.7935  std=0.0273
RandomForest            mean=0.8076  std=0.0212
XGBoost                 mean=0.8006  std=0.0239


In [5]:
# ── 4. Select best model, final training + hold-out testing ────────
best_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f"\n[INFO] Best model: {best_name}")

best_model = models[best_name]
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived'])

print(f"\nHold-out Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(report)



[INFO] Best model: RandomForest

Hold-out Accuracy: 0.8156

Classification Report:
              precision    recall  f1-score   support

Not Survived       0.83      0.87      0.85       110
    Survived       0.78      0.72      0.75        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.82      0.81       179



In [6]:
# ── 5. Save model ────────────────────────────────────────────
os.makedirs('../models', exist_ok=True)
model_path = f'../models/{best_name.lower().replace(" ", "_")}.joblib'
joblib.dump(best_model, model_path)
print(f"[PASS] Model saved: {model_path}")


[PASS] Model saved: ../models/randomforest.joblib


In [7]:
# ── 6. Write metrics.txt ────────────────────────────────────
os.makedirs('../results', exist_ok=True)
metrics_path = '../results/metrics.txt'

with open(metrics_path, 'w',encoding='utf-8') as f:
    f.write("── CV (5-Fold Accuracy)──\n")
    for name, scores in cv_results.items():
        f.write(f"{name:22s}  mean={scores.mean():.4f}  std={scores.std():.4f}\n")
    f.write(f"\nBest model: {best_name}\n")
    f.write(f"Hold-out Accuracy: {acc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(report)

print(f"[PASS] Metrics written to: {metrics_path}")


[PASS] Metrics written to: ../results/metrics.txt


In [8]:
# ── 7. Assertion tests ────────────────────────────────────────────
print("\n── Assertion tests ──")

assert acc > 0.75, \
    f"[FAIL] Accuracy too low: {acc:.4f}"
print(f"[PASS] Accuracy meets target: {acc:.4f} > 0.75")

assert os.path.exists(model_path), \
    f"[FAIL] Model file not found: {model_path}"
print(f"[PASS] Model file exists: {model_path}")

assert os.path.exists(metrics_path), \
    f"[FAIL] metrics.txt not found"
print(f"[PASS] metrics.txt exists: {metrics_path}")

assert X_test.shape[1] == X_train.shape[1], \
    "[FAIL] Feature count mismatch between test and train sets"
print(f"[PASS] Feature count consistent: {X_test.shape[1]} columns")



── Assertion tests ──
[PASS] Accuracy meets target: 0.8156 > 0.75
[PASS] Model file exists: ../models/randomforest.joblib
[PASS] metrics.txt exists: ../results/metrics.txt
[PASS] Feature count consistent: 8 columns
